# BandSplitVAE End-to-End (Colab)

Pretrain → fine-tune (margin, discriminator) → evaluate → visualize. Runs on Colab with local checkpoints/output; dataset zip is copied from Drive, extracted locally, then the zip is removed.

In [ ]:
import os
import sys
import shutil
from pathlib import Path

use_colab = "google.colab" in sys.modules
if use_colab:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)

PRETRAIN_DATASET = os.environ.get("PRETRAIN_DATASET", "20GBprocessed")
FINETUNE_DATASET = os.environ.get("FINETUNE_DATASET", "processed")
DRIVE_PROJECT = Path("/content/drive/MyDrive/liveness_detection_vae")
project_dir = DRIVE_PROJECT if DRIVE_PROJECT.exists() else Path.cwd()
LOCAL_DATA = Path("/content/datasets")
LOCAL_RUNS = Path("/content/runs")
LOCAL_DATA.mkdir(parents=True, exist_ok=True)
LOCAL_RUNS.mkdir(parents=True, exist_ok=True)

print("Project dir:", project_dir)
print("Local data:", LOCAL_DATA)
print("Local runs:", LOCAL_RUNS)
print("Pretrain dataset:", PRETRAIN_DATASET)
print("Finetune dataset:", FINETUNE_DATASET)


In [ ]:
lib_dir = project_dir / "lib"
sys.path.insert(0, str(lib_dir))
sys.path.insert(0, str(project_dir))

from model import setup_data_paths

drive_root = project_dir / "datasets"
LOCAL_DATA.mkdir(parents=True, exist_ok=True)

pretrain_data_dir, finetune_root, stage2_train_pairs, stage2_val_pairs = setup_data_paths(
    PRETRAIN_DATASET,
    FINETUNE_DATASET,
    drive_root,
    LOCAL_DATA,
)

os.environ["BANDVAE_DATA_DIR"] = str(pretrain_data_dir)
print("Project dir:", project_dir)
print("Pretrain data dir:", pretrain_data_dir)
print("Finetune data root:", finetune_root)
print("Stage2 train count:", len(stage2_train_pairs), "val count:", len(stage2_val_pairs))
print("Runs dir:", LOCAL_RUNS)
os.chdir(project_dir)


## Stage 1: Pretrain (one-class VAE)

In [ ]:
import time
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

from config_bandvae import get_config
from dataset_bandvae import LipLivenessBandDataset
from model import BandSplitVAE, band_split_vae_loss, train_epoch_stage1, validate_stage1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp_stage1 = device.type == "cuda"
train_dtype_stage1 = torch.float16 if use_amp_stage1 else torch.float32

config_stage1 = get_config("full")
config_stage1.T_fixed = 300
config_stage1.fc_low = 2.0
config_stage1.fc_high = 8.0
config_stage1.filter_order = 4
config_stage1.C_h = 48
config_stage1.C_z = 12
config_stage1.dilations = [1, 2, 4]
config_stage1.lr = 1e-3
config_stage1.epochs = 20
config_stage1.batch_size = 64
config_stage1.num_workers = 4
config_stage1.data_dir = str(pretrain_data_dir)
config_stage1.device = device
config_stage1.dtype = train_dtype_stage1
stage1_dir = LOCAL_RUNS / "stage1_pretrain"
stage1_dir.mkdir(parents=True, exist_ok=True)
stage1_ckpt = stage1_dir / "stage1_pretrained.pt"
print(config_stage1)


In [ ]:
dataset_dtype = train_dtype_stage1

stage1_dataset = LipLivenessBandDataset(
    data_dir=config_stage1.data_dir,
    T_fixed=config_stage1.T_fixed,
    fps=config_stage1.fps,
    use_procrustes=True,
    use_acceleration=config_stage1.use_acceleration,
    use_angle=config_stage1.use_angle,
    use_angle_rate=config_stage1.use_angle_rate,
    fc_low=config_stage1.fc_low,
    fc_high=config_stage1.fc_high,
    filter_order=config_stage1.filter_order,
    random_crop=True,
    processing_dtype=torch.float32,
    output_dtype=dataset_dtype,
)

train_size = int((1 - config_stage1.val_split) * len(stage1_dataset))
val_size = len(stage1_dataset) - train_size
stage1_train, stage1_val = random_split(
    stage1_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42),
)

stage1_train_loader = DataLoader(
    stage1_train,
    batch_size=config_stage1.batch_size,
    shuffle=True,
    num_workers=config_stage1.num_workers,
    pin_memory=True,
)
stage1_val_loader = DataLoader(
    stage1_val,
    batch_size=config_stage1.batch_size,
    shuffle=False,
    num_workers=config_stage1.num_workers,
    pin_memory=True,
)

print(f"Train: {len(stage1_train)}, Val: {len(stage1_val)}")

In [ ]:
stage1_model = BandSplitVAE(
    C_in_per_band=config_stage1.C_in_per_band,
    C_h=config_stage1.C_h,
    C_z=config_stage1.C_z,
    dilations=config_stage1.dilations,
).to(config_stage1.device)

stage1_optimizer = optim.Adam(stage1_model.parameters(), lr=config_stage1.lr)
scaler_stage1 = torch.cuda.amp.GradScaler(enabled=use_amp_stage1)


In [ ]:
best_val = float("inf")
for epoch in range(1, config_stage1.epochs + 1):
    start = time.time()
    train_loss = train_epoch_stage1(stage1_model, stage1_train_loader, stage1_optimizer, config_stage1.device)
    val_loss = validate_stage1(stage1_model, stage1_val_loader, config_stage1.device)
    duration = (time.time() - start) / 60
    print(
        f"Epoch {epoch}/{config_stage1.epochs} | "
        f"train {train_loss['total']:.4f} (lf {train_loss['lf_rec']:.4f}, bp {train_loss['bp_rec']:.4f}, hf {train_loss['hf_rec']:.4f}) | "
        f"val {val_loss['total']:.4f} (lf {val_loss['lf_rec']:.4f}, bp {val_loss['bp_rec']:.4f}, hf {val_loss['hf_rec']:.4f}) | "
        f"{duration:.1f} min"
    )
    if val_loss["total"] < best_val:
        best_val = val_loss["total"]
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": stage1_model.state_dict(),
                "optimizer_state_dict": stage1_optimizer.state_dict(),
                "val_loss": best_val,
                "config": config_stage1,
            },
            stage1_ckpt,
        )
        print(f"Saved best to {stage1_ckpt}")

## Stage 2 data loaders

In [ ]:
from torch.utils.data import DataLoader
from dataset_stage2 import Stage2Dataset

if stage2_train_pairs is None or stage2_val_pairs is None:
    raise RuntimeError("Stage 2 file lists not prepared; ensure preprocessed dataset is available.")

config_stage2 = get_config("full")
config_stage2.T_fixed = 300
config_stage2.fc_low = 2.0
config_stage2.fc_high = 8.0
config_stage2.filter_order = 4
config_stage2.C_h = 48
config_stage2.C_z = 12
config_stage2.dilations = [1, 2, 4]
config_stage2.lr = 1e-4
config_stage2.epochs = 10
config_stage2.batch_size = 64
config_stage2.num_workers = 4
config_stage2.device = device

stage2_train_dataset = Stage2Dataset(
    split_json_path=None,
    split_name=None,
    T_fixed=config_stage2.T_fixed,
    fps=config_stage2.fps,
    use_acceleration=config_stage2.use_acceleration,
    use_angle=config_stage2.use_angle,
    use_angle_rate=config_stage2.use_angle_rate,
    fc_low=config_stage2.fc_low,
    fc_high=config_stage2.fc_high,
    filter_order=config_stage2.filter_order,
    random_crop=True,
    files_and_labels=stage2_train_pairs,
)

stage2_val_dataset = Stage2Dataset(
    split_json_path=None,
    split_name=None,
    T_fixed=config_stage2.T_fixed,
    fps=config_stage2.fps,
    use_acceleration=config_stage2.use_acceleration,
    use_angle=config_stage2.use_angle,
    use_angle_rate=config_stage2.use_angle_rate,
    fc_low=config_stage2.fc_low,
    fc_high=config_stage2.fc_high,
    filter_order=config_stage2.filter_order,
    random_crop=False,
    files_and_labels=stage2_val_pairs,
)

stage2_train_loader = DataLoader(
    stage2_train_dataset,
    batch_size=config_stage2.batch_size,
    shuffle=True,
    num_workers=config_stage2.num_workers,
    pin_memory=True,
)
stage2_val_loader = DataLoader(
    stage2_val_dataset,
    batch_size=config_stage2.batch_size,
    shuffle=False,
    num_workers=config_stage2.num_workers,
    pin_memory=True,
)

print("Stage2 train samples:", len(stage2_train_dataset), "Val samples:", len(stage2_val_dataset))

## Stage 2: Fine-tune (margin)

In [ ]:
from model import margin_loss, train_epoch_margin, validate_margin
import torch.optim as optim

MARGIN = 0.5
LAMBDA_MARGIN = 1.0
stage2_margin_dir = LOCAL_RUNS / "stage2_margin"
stage2_margin_dir.mkdir(parents=True, exist_ok=True)
stage2_margin_ckpt = stage2_margin_dir / "stage2_margin_best.pt"

checkpoint_stage1 = torch.load(stage1_ckpt, map_location=config_stage2.device)
margin_model = BandSplitVAE(
    C_in_per_band=config_stage2.C_in_per_band,
    C_h=config_stage2.C_h,
    C_z=config_stage2.C_z,
    dilations=config_stage2.dilations,
).to(config_stage2.device)
margin_model.load_state_dict(checkpoint_stage1["model_state_dict"])
margin_optimizer = optim.Adam(margin_model.parameters(), lr=config_stage2.lr)
use_amp_stage2 = config_stage2.device == "cuda"
train_dtype_stage2 = torch.float16 if use_amp_stage2 else torch.float32
scaler_margin = torch.cuda.amp.GradScaler(enabled=use_amp_stage2)


In [ ]:
best_sep = -float("inf")
for epoch in range(1, config_stage2.epochs + 1):
    start = time.time()
    train_loss = train_epoch_margin(
        margin_model,
        stage2_train_loader,
        margin_optimizer,
        config_stage2.device,
        train_dtype_stage2,
        use_amp_stage2,
        MARGIN,
        LAMBDA_MARGIN,
        scaler_margin,
    )
    val_loss = validate_margin(
        margin_model,
        stage2_val_loader,
        config_stage2.device,
        train_dtype_stage2,
        use_amp_stage2,
        MARGIN,
    )
    duration = (time.time() - start) / 60
    print(
        f"Epoch {epoch}/{config_stage2.epochs} | "
        f"train total {train_loss['total']:.4f} real {train_loss['real_rec']:.4f} fake {train_loss['fake_rec']:.4f} sep {train_loss['separation']:.4f} | "
        f"val total {val_loss['total']:.4f} real {val_loss['real_rec']:.4f} fake {val_loss['fake_rec']:.4f} sep {val_loss['separation']:.4f} | "
        f"{duration:.1f} min"
    )
    if val_loss["separation"] > best_sep:
        best_sep = val_loss["separation"]
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": margin_model.state_dict(),
                "optimizer_state_dict": margin_optimizer.state_dict(),
                "val_metrics": val_loss,
                "config": config_stage2,
                "margin": MARGIN,
                "lambda_margin": LAMBDA_MARGIN,
            },
            stage2_margin_ckpt,
        )
        print(f"Saved best to {stage2_margin_ckpt}")


## Stage 2: Fine-tune (latent discriminator)

In [ ]:
from model import LatentDiscriminator, train_epoch_disc, validate_disc
import torch.nn as nn

disc_dir = LOCAL_RUNS / "stage2_discriminator"
disc_dir.mkdir(parents=True, exist_ok=True)
stage2_disc_ckpt = disc_dir / "stage2_discriminator_best.pt"

checkpoint_stage1 = torch.load(stage1_ckpt, map_location=config_stage2.device)
disc_model = BandSplitVAE(
    C_in_per_band=config_stage2.C_in_per_band,
    C_h=config_stage2.C_h,
    C_z=config_stage2.C_z,
    dilations=config_stage2.dilations,
).to(config_stage2.device)
disc_model.load_state_dict(checkpoint_stage1["model_state_dict"])

discriminator = LatentDiscriminator(config_stage2.C_z).to(config_stage2.device)
disc_optimizer = optim.Adam(list(disc_model.parameters()) + list(discriminator.parameters()), lr=config_stage2.lr)
criterion_cls = nn.BCEWithLogitsLoss()
scaler_disc = torch.cuda.amp.GradScaler(enabled=use_amp_stage2)


In [ ]:
best_disc = float("inf")
for epoch in range(1, config_stage2.epochs + 1):
    start = time.time()
    train_loss = train_epoch_disc(
        disc_model,
        discriminator,
        stage2_train_loader,
        disc_optimizer,
        config_stage2.device,
        train_dtype_stage2,
        use_amp_stage2,
        criterion_cls,
        scaler_disc,
    )
    val_loss = validate_disc(
        disc_model,
        discriminator,
        stage2_val_loader,
        config_stage2.device,
        train_dtype_stage2,
        use_amp_stage2,
        criterion_cls,
    )
    duration = (time.time() - start) / 60
    print(
        f"Epoch {epoch}/{config_stage2.epochs} | "
        f"train total {train_loss['total']:.4f} rec {train_loss['rec']:.4f} cls {train_loss['cls']:.4f} acc {train_loss['accuracy']:.4f} | "
        f"val total {val_loss['total']:.4f} rec {val_loss['rec']:.4f} cls {val_loss['cls']:.4f} acc {val_loss['accuracy']:.4f} | "
        f"{duration:.1f} min"
    )
    if val_loss["total"] < best_disc:
        best_disc = val_loss["total"]
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": disc_model.state_dict(),
                "optimizer_state_dict": disc_optimizer.state_dict(),
                "discriminator_state_dict": discriminator.state_dict(),
                "val_metrics": val_loss,
                "config": config_stage2,
            },
            stage2_disc_ckpt,
        )
        print(f"Saved best to {stage2_disc_ckpt}")


In [ ]:
best_acc = 0.0
for epoch in range(1, config_stage2.epochs + 1):
    start = time.time()
    train_loss = train_epoch_disc(disc_model, discriminator, stage2_train_loader, disc_optimizer, config_stage2.device)
    val_loss = validate_disc(disc_model, discriminator, stage2_val_loader, config_stage2.device)
    duration = (time.time() - start) / 60
    print(
        f"Epoch {epoch}/{config_stage2.epochs} | "
        f"train total {train_loss['total']:.4f} rec {train_loss['rec']:.4f} cls {train_loss['cls']:.4f} acc {train_loss['accuracy']:.4f} | "
        f"val total {val_loss['total']:.4f} rec {val_loss['rec']:.4f} cls {val_loss['cls']:.4f} acc {val_loss['accuracy']:.4f} | "
        f"{duration:.1f} min"
    )
    if val_loss["accuracy"] > best_acc:
        best_acc = val_loss["accuracy"]
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": disc_model.state_dict(),
                "discriminator_state_dict": discriminator.state_dict(),
                "optimizer_state_dict": disc_optimizer.state_dict(),
                "val_metrics": val_loss,
                "config": config_stage2,
            },
            stage2_disc_ckpt,
        )
        print(f"Saved best to {stage2_disc_ckpt}")

## Evaluation (two-stage Gaussian filtering with PCA)

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
from torch.utils.data import DataLoader

EVAL_CHECKPOINT = stage2_margin_ckpt
PCA_DIM = 10
CONFIDENCE_STAGE1 = 0.95
CONFIDENCE_STAGE2 = 0.70
eval_output_dir = LOCAL_RUNS / "evaluation_main"
eval_output_dir.mkdir(parents=True, exist_ok=True)

train_eval_dataset = Stage2Dataset(
    split_json_path=None,
    split_name=None,
    T_fixed=config_stage2.T_fixed,
    fps=config_stage2.fps,
    use_acceleration=config_stage2.use_acceleration,
    use_angle=config_stage2.use_angle,
    use_angle_rate=config_stage2.use_angle_rate,
    fc_low=config_stage2.fc_low,
    fc_high=config_stage2.fc_high,
    filter_order=config_stage2.filter_order,
    random_crop=False,
    files_and_labels=stage2_train_pairs,
)
train_eval_loader = DataLoader(train_eval_dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=False)

test_eval_dataset = Stage2Dataset(
    split_json_path=None,
    split_name=None,
    T_fixed=config_stage2.T_fixed,
    fps=config_stage2.fps,
    use_acceleration=config_stage2.use_acceleration,
    use_angle=config_stage2.use_angle,
    use_angle_rate=config_stage2.use_angle_rate,
    fc_low=config_stage2.fc_low,
    fc_high=config_stage2.fc_high,
    filter_order=config_stage2.filter_order,
    random_crop=False,
    files_and_labels=stage2_val_pairs,
)
test_eval_loader = DataLoader(test_eval_dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=False)

sns.set_theme(style="whitegrid")

In [ ]:
from eval_utils import (
    compute_bandwise_reconstruction_losses,
    extract_latent_vectors,
    fit_gaussian_multivariate,
    is_within_confidence_multivariate,
    apply_pca,
    two_stage_filtering_pca10d,
    plot_confusion_matrix,
    plot_detailed_metrics,
    run_evaluation,
)

In [ ]:
# Evaluation runs are triggered below using eval_utils.run_evaluation

In [ ]:
main_eval = run_evaluation(
    EVAL_CHECKPOINT,
    train_eval_loader,
    test_eval_loader,
    device,
    eval_output_dir,
    pca_dim=PCA_DIM,
    confidence_stage1=CONFIDENCE_STAGE1,
    confidence_stage2=CONFIDENCE_STAGE2,
)


## Visualize checkpoints

In [ ]:

visualize_dir = LOCAL_RUNS / "visualizations"
visualize_dir.mkdir(parents=True, exist_ok=True)
checkpoints_to_visualize = [stage2_margin_ckpt, stage2_disc_ckpt]
results_by_ckpt = {}
for ckpt in checkpoints_to_visualize:
    if not ckpt.exists():
        print("Skip missing checkpoint", ckpt)
        continue
    ckpt_dir = visualize_dir / ckpt.stem
    res = run_evaluation(
        ckpt,
        train_eval_loader,
        test_eval_loader,
        device,
        ckpt_dir,
        pca_dim=PCA_DIM,
        confidence_stage1=CONFIDENCE_STAGE1,
        confidence_stage2=CONFIDENCE_STAGE2,
    )
    results_by_ckpt[str(ckpt)] = res
print("Collected results for", len(results_by_ckpt), "checkpoints")
